<img src='../images/xebia-logo.png' width='300px' align='right' style="padding: 15px">

# Day 3 Morning: Review Session

This notebook brings together everything from the past two days — packaging, code quality, testing, logging, and OOP — into a single hands-on session.

You'll be building a small but complete **Polars-based data pipeline** for the animal shelter dataset, applying all the patterns you've practised.

**Don't worry about Polars syntax** — every cell that needs it has the Polars code already written. Your job is to apply the Python patterns around it.

---

### Topics covered

| # | Topic | From day |
|---|---|---|
| 1 | Packaging & imports | Day 1 — notebook 01 |
| 2 | Code quality & refactoring | Day 1 — notebook 02 |
| 3 | Unit tests & fixtures | Day 1 — notebooks 04, 05 |
| 4 | Logging | Day 2 — notebook 06 |
| 5 | Context managers | Day 2 — OOP track |
| 6 | Classes, inheritance & dunders | Day 2 — OOP track |
| 7 | Putting it all together | Today |

---

## Setup

Run this cell first. It installs any missing packages and imports everything needed for the session.

In [1]:
# Run once to make sure all packages are available
import subprocess, sys
packages = ["polars", "pytest", "pytest-ipynb2", "hypothesis"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

In [2]:
import logging
import re
import time
from contextlib import contextmanager
from functools import wraps
from pathlib import Path

import polars as pl

print("All imports OK  ✓")
print(f"Polars version: {pl.__version__}")

All imports OK  ✓
Polars version: 1.39.3


---

## Part 1 — Polars orientation (10 min)

Here is a quick tour of the Polars operations you'll see today. Read through and run each cell.

In [3]:
# Creating a DataFrame — like pd.DataFrame(...)
df = pl.DataFrame({
    "name":   ["Luna", "Biscuit", "Rex", "Shadow"],
    "type":   ["Cat",  "Dog",    "Dog",  "Cat"],
    "age_yr": [2,      5,        1,      3],
})
df

name,type,age_yr
str,str,i64
"""Luna""","""Cat""",2
"""Biscuit""","""Dog""",5
"""Rex""","""Dog""",1
"""Shadow""","""Cat""",3


In [4]:
# .filter() — like df[df['col'] == value]
df.filter(pl.col("type") == "Dog")

name,type,age_yr
str,str,i64
"""Biscuit""","""Dog""",5
"""Rex""","""Dog""",1


In [5]:
# .with_columns() — add or replace a column
# Polars DataFrames are immutable: this returns a NEW DataFrame, it does not change df
df.with_columns(
    (pl.col("age_yr") * 365).alias("age_days")
)

name,type,age_yr,age_days
str,str,i64,i64
"""Luna""","""Cat""",2,730
"""Biscuit""","""Dog""",5,1825
"""Rex""","""Dog""",1,365
"""Shadow""","""Cat""",3,1095


In [6]:
# .select() — choose columns (like df[['col1', 'col2']])
df.select(["name", "type"])

name,type
str,str
"""Luna""","""Cat"""
"""Biscuit""","""Dog"""
"""Rex""","""Dog"""
"""Shadow""","""Cat"""


In [7]:
# Chaining — this is the idiomatic Polars style
result = (
    df
    .filter(pl.col("age_yr") >= 2)
    .with_columns((pl.col("age_yr") * 365).alias("age_days"))
    .select(["name", "type", "age_days"])
)
result

name,type,age_days
str,str,i64
"""Luna""","""Cat""",730
"""Biscuit""","""Dog""",1825
"""Shadow""","""Cat""",1095


In [8]:
# .pipe() — pass a DataFrame through a function, exactly like pandas
def add_senior_flag(df: pl.DataFrame) -> pl.DataFrame:
    """Mark animals older than 7 years as senior."""
    return df.with_columns(
        (pl.col("age_yr") > 7).cast(pl.Int8).alias("is_senior")
    )

df.pipe(add_senior_flag)

name,type,age_yr,is_senior
str,str,i64,i8
"""Luna""","""Cat""",2,0
"""Biscuit""","""Dog""",5,0
"""Rex""","""Dog""",1,0
"""Shadow""","""Cat""",3,0


That's all the Polars you need today. Everything else is Python.

---

## Part 2 — Refactoring (25 min)

Below is a working but messy data pipeline written in Polars. It does the same thing as the original `add_features()` from Day 1 — but it's all in one place and it mutates nothing (Polars DataFrames are immutable, so this is free!).

Your task: **refactor it** into small, well-named functions.

In [ ]:
# ── The messy version ─────────────────────────────────────────────────────────
# Read this, understand what it does, then refactor it below.

def add_all_features_messy(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .with_columns(
            # is_dog
            (pl.col("animal_type").str.to_lowercase() == "dog")
            .cast(pl.Int8).alias("is_dog")
        )
        .with_columns(
            # has_name
            (pl.col("name").str.to_lowercase() != "unknown")
            .alias("has_name")
        )
        .with_columns(
            # sex
            pl.when(pl.col("sex_upon_outcome").str.ends_with("Female")).then(pl.lit("female"))
            .when(pl.col("sex_upon_outcome").str.ends_with("Male")).then(pl.lit("male"))
            .otherwise(pl.lit("unknown"))
            .alias("sex")
        )
        .with_columns(
            # neutered
            pl.when(pl.col("sex_upon_outcome").str.to_lowercase().str.contains("neutered|spayed"))
            .then(pl.lit("fixed"))
            .when(pl.col("sex_upon_outcome").str.to_lowercase().str.contains("intact"))
            .then(pl.lit("intact"))
            .otherwise(pl.lit("unknown"))
            .alias("neutered")
        )
        .with_columns(
            # age_days
            pl.when(pl.col("age_upon_outcome").str.contains("year"))
            .then(pl.col("age_upon_outcome").str.extract(r"(\d+)").cast(pl.Int32) * 365)
            .when(pl.col("age_upon_outcome").str.contains("month"))
            .then(pl.col("age_upon_outcome").str.extract(r"(\d+)").cast(pl.Int32) * 30)
            .when(pl.col("age_upon_outcome").str.contains("week"))
            .then(pl.col("age_upon_outcome").str.extract(r"(\d+)").cast(pl.Int32) * 7)
            .otherwise(pl.lit(1))
            .alias("age_days")
        )
    )

### <mark>Exercise 2a: Split into helper functions</mark>

Fill in each function below. The Polars expression for each feature is already written — you just need to put it in the right place.

Each function should:
- Accept a `pl.Series` (a single column) and return a `pl.Series`
- Do exactly one thing
- Have a leading underscore if it's only used inside `add_features()`

In [ ]:
def _check_is_dog(animal_type: pl.Series) -> pl.Series:
    """Return 1 if dog, 0 otherwise."""
    # Polars expression — already written for you:
    # (animal_type.str.to_lowercase() == "dog").cast(pl.Int8)
    pass  # TODO: replace with the expression above


def _check_has_name(name: pl.Series) -> pl.Series:
    """Return True if name is not 'unknown'."""
    # Polars expression — already written for you:
    # name.str.to_lowercase() != "unknown"
    pass  # TODO


def _get_sex(sex_upon_outcome: pl.Series) -> pl.Series:
    """Return 'male', 'female', or 'unknown'."""
    # Polars expression:
    # pl.when(sex_upon_outcome.str.ends_with("Female")).then(pl.lit("female"))
    # .when(sex_upon_outcome.str.ends_with("Male")).then(pl.lit("male"))
    # .otherwise(pl.lit("unknown"))
    pass  # TODO


def _get_neutered(sex_upon_outcome: pl.Series) -> pl.Series:
    """Return 'fixed', 'intact', or 'unknown'."""
    # Polars expression:
    # pl.when(sex_upon_outcome.str.to_lowercase().str.contains("neutered|spayed")).then(pl.lit("fixed"))
    # .when(sex_upon_outcome.str.to_lowercase().str.contains("intact")).then(pl.lit("intact"))
    # .otherwise(pl.lit("unknown"))
    pass  # TODO


def _compute_age_days(age_upon_outcome: pl.Series) -> pl.Series:
    """Convert age string like '2 years' or '3 months' to number of days."""
    # Polars expression:
    # pl.when(age_upon_outcome.str.contains("year"))
    # .then(age_upon_outcome.str.extract(r"(\d+)").cast(pl.Int32) * 365)
    # .when(age_upon_outcome.str.contains("month"))
    # .then(age_upon_outcome.str.extract(r"(\d+)").cast(pl.Int32) * 30)
    # .when(age_upon_outcome.str.contains("week"))
    # .then(age_upon_outcome.str.extract(r"(\d+)").cast(pl.Int32) * 7)
    # .otherwise(pl.lit(1))
    pass  # TODO


def add_features(df: pl.DataFrame) -> pl.DataFrame:
    """Add all engineered features to the DataFrame.
    
    This is the only public function — callers use this, not the helpers.
    """
    # TODO: call each helper function and use .with_columns() to add results
    # Hint: use df.with_columns(helper(df["col"]).alias("new_col_name"))
    pass

In [ ]:
# ── Test your refactored functions with this small DataFrame ──────────────────
sample = pl.DataFrame({
    "animal_type":      ["Dog",           "Cat",           "Dog"],
    "name":             ["Biscuit",        "unknown",       "Rex"],
    "sex_upon_outcome": ["Neutered Male",  "Intact Female", "Spayed Female"],
    "age_upon_outcome": ["2 years",        "6 months",      "3 weeks"],
    "outcome_type":     ["Adoption",       "Transfer",      "Return to Owner"],
})

result = add_features(sample)
result

**Expected output — check yours matches:**

| name | is_dog | has_name | sex | neutered | age_days |
|------|--------|----------|-----|----------|----------|
| Biscuit | 1 | True | male | fixed | 730 |
| unknown | 0 | False | female | intact | 180 |
| Rex | 1 | True | female | fixed | 21 |

---

## Part 3 — Unit tests & fixtures (30 min)

### <mark>Exercise 3a: Write unit tests for your helper functions</mark>

The fixture `sex_series` is provided. Write at least one test per helper function.

Remember:
- Test the **happy path** (normal input)
- Test at least one **edge case** (e.g. what happens with `"Unknown"` input?)
- Use `assert result.to_list() == [...]` or `assert result.equals(expected)` for Polars Series

In [ ]:
import pytest

# ── Fixtures ──────────────────────────────────────────────────────────────────
# These are provided. Use them in your tests by passing them as arguments.

@pytest.fixture
def sex_series():
    return pl.Series(["Neutered Male", "Spayed Female", "Intact Male", "Unknown"])

@pytest.fixture  
def age_series():
    return pl.Series(["2 years", "6 months", "3 weeks", "1 day"])


# ── Tests ─────────────────────────────────────────────────────────────────────

def test_check_is_dog():
    animal_types = pl.Series(["Dog", "Cat", "dog", "Bird"])
    result = _check_is_dog(animal_types)
    assert result.to_list() == [1, 0, 1, 0]  # case-insensitive


def test_check_has_name():
    # TODO: test that "unknown" → False and a real name → True
    pass


def test_get_sex_male(sex_series):
    # TODO: test that "Neutered Male" maps to "male"
    pass


def test_get_sex_female(sex_series):
    # TODO: test that "Spayed Female" maps to "female"
    pass


def test_get_sex_unknown(sex_series):
    # TODO: test that "Unknown" maps to "unknown"
    pass


def test_get_neutered_fixed(sex_series):
    # TODO: test that "Neutered Male" and "Spayed Female" both map to "fixed"
    pass


def test_get_neutered_intact(sex_series):
    # TODO: test that "Intact Male" maps to "intact"
    pass


def test_compute_age_days_years(age_series):
    result = _compute_age_days(age_series)
    assert result[0] == 730  # 2 years = 730 days


def test_compute_age_days_months(age_series):
    # TODO: check that 6 months = 180 days
    pass


def test_compute_age_days_weeks(age_series):
    # TODO: check that 3 weeks = 21 days
    pass


def test_add_features_output_columns():
    """add_features() must produce all expected columns."""
    df = pl.DataFrame({
        "animal_type":      ["Dog"],
        "name":             ["Biscuit"],
        "sex_upon_outcome": ["Neutered Male"],
        "age_upon_outcome": ["2 years"],
        "outcome_type":     ["Adoption"],
    })
    result = add_features(df)
    for col in ["is_dog", "has_name", "sex", "neutered", "age_days"]:
        assert col in result.columns, f"Missing column: {col}"


def test_add_features_no_mutation():
    """add_features() must not modify the original DataFrame."""
    df = pl.DataFrame({
        "animal_type":      ["Dog"],
        "name":             ["Biscuit"],
        "sex_upon_outcome": ["Neutered Male"],
        "age_upon_outcome": ["2 years"],
        "outcome_type":     ["Adoption"],
    })
    original_cols = df.columns.copy()
    _ = add_features(df)
    assert df.columns == original_cols  # Polars is immutable — this should always pass!

In [ ]:
# Run your tests directly in the notebook
# You should see all tests pass (or get clear error messages for the TODOs)
!python -m pytest {__vsc_ipynb_file__ if '__vsc_ipynb_file__' in dir() else 'day3_morning_review.ipynb'} -v 2>/dev/null || pytest.main(["-v", "--no-header", "-x"])

---

## Part 4 — Logging (20 min)

### <mark>Exercise 4a: Add logging to `add_features`</mark>

Modify your `add_features()` function to log:
- At `INFO` level: how many rows are being processed
- At `DEBUG` level: the column names of the input DataFrame
- At `INFO` level: a confirmation message when done

In [ ]:
# ── Logger setup — provided, just run this ───────────────────────────────────
def setup_logger(level: int = logging.INFO) -> None:
    """Set up the animal_shelter logger."""
    logger = logging.getLogger("animal_shelter")
    if logger.handlers:
        logger.handlers.clear()
    logger.setLevel(level)
    handler = logging.StreamHandler()
    handler.setLevel(level)
    handler.setFormatter(logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    ))
    logger.addHandler(handler)

setup_logger(level=logging.DEBUG)

# Get the logger — use this in your functions
logger = logging.getLogger("animal_shelter")

In [ ]:
def add_features(df: pl.DataFrame) -> pl.DataFrame:
    """Add all engineered features to the DataFrame."""
    # TODO: add log messages at INFO and DEBUG level
    # logger.info("Processing %d rows", df.height)
    # logger.debug("Input columns: %s", df.columns)
    
    result = (
        df
        .with_columns(_check_is_dog(df["animal_type"]).alias("is_dog"))
        .with_columns(_check_has_name(df["name"]).alias("has_name"))
        .with_columns(_get_sex(df["sex_upon_outcome"]).alias("sex"))
        .with_columns(_get_neutered(df["sex_upon_outcome"]).alias("neutered"))
        .with_columns(_compute_age_days(df["age_upon_outcome"]).alias("age_days"))
    )
    
    # TODO: log a completion message
    return result


# Test it — you should see log output
add_features(sample)

### <mark>Exercise 4b: Write a `log_step` decorator</mark>

Write a decorator called `log_step` that, when applied to a pipeline function:
1. Logs the function name and input row count at `INFO` level
2. Runs the function
3. Logs the output row count (and how many rows were dropped, if any)

This is the same pattern from the `Modern_Pipelines_Polars.ipynb` notebook.

In [ ]:
def log_step(func):
    """Decorator that logs input/output row counts for a pipeline step."""
    @wraps(func)
    def wrapper(df: pl.DataFrame, *args, **kwargs) -> pl.DataFrame:
        # TODO: log function name and input row count
        result = func(df, *args, **kwargs)
        # TODO: log output row count and rows dropped
        return result
    return wrapper


# Apply your decorator to a pipeline step
@log_step
def remove_unknown_outcomes(df: pl.DataFrame) -> pl.DataFrame:
    """Drop rows where outcome_type is unknown."""
    return df.filter(pl.col("outcome_type") != "Unknown")


@log_step
def remove_missing_ages(df: pl.DataFrame) -> pl.DataFrame:
    """Drop rows where age is missing or 'Unknown'."""
    return df.filter(pl.col("age_upon_outcome") != "Unknown")


# Test the decorator
test_df = pl.DataFrame({
    "animal_type":      ["Dog", "Cat",     "Dog",    "Cat"],
    "name":             ["Rex", "unknown", "Biscuit", "Luna"],
    "sex_upon_outcome": ["Neutered Male", "Intact Female", "Spayed Female", "Unknown"],
    "age_upon_outcome": ["2 years", "Unknown", "6 months", "1 year"],
    "outcome_type":     ["Adoption", "Transfer", "Unknown", "Return to Owner"],
})

cleaned = (
    test_df
    .pipe(remove_unknown_outcomes)
    .pipe(remove_missing_ages)
)
# You should see log output showing rows going from 4 → 3 → 2
cleaned

---

## Part 5 — Context managers (20 min)

### <mark>Exercise 5a: A `timer` context manager</mark>

Write a context manager called `timer` that measures how long a block of code takes and logs the result.

Use the `@contextmanager` generator approach from the OOP notebooks.

In [ ]:
@contextmanager
def timer(description: str = "block"):
    """Context manager that logs the execution time of the enclosed block.
    
    Usage:
        with timer("preprocessing"):
            result = add_features(df)
    """
    # TODO:
    # 1. Record the start time (use time.perf_counter())
    # 2. yield
    # 3. Record the end time and log the elapsed time
    # Hint: use try/finally so the time is always logged even if an exception is raised
    pass


# Test it
with timer("feature engineering"):
    result = add_features(sample)

# You should see a log line like:
# INFO - feature engineering took 0.0012s

### <mark>Exercise 5b: A `DataFrameCheckpoint` context manager</mark>

Write a context manager called `DataFrameCheckpoint` that:
1. On entry: saves a snapshot of a DataFrame to a temporary CSV file and logs where it saved it
2. Yields the path to that file (so the `as` variable gives you the path)
3. On exit: deletes the temporary file and logs that it was cleaned up

This is useful in ML pipelines when you want to inspect intermediate data without keeping it around permanently.

In [ ]:
import tempfile
import os

@contextmanager
def dataframe_checkpoint(df: pl.DataFrame, name: str = "checkpoint"):
    """Save a DataFrame snapshot to a temp file, yield the path, then clean up.
    
    Usage:
        with dataframe_checkpoint(my_df, "after_cleaning") as path:
            print(f"Inspect at: {path}")
            # file exists here
        # file is deleted here
    """
    # TODO:
    # 1. Create a temp file with suffix .csv (use tempfile.NamedTemporaryFile)
    # 2. Save df to that file using df.write_csv(path)
    # 3. Log where you saved it
    # 4. yield the path
    # 5. In the finally block: delete the file and log that you did
    pass


# Test it
with dataframe_checkpoint(sample, "sample_data") as path:
    print(f"File exists during context: {Path(path).exists()}")
    reloaded = pl.read_csv(path)
    print(f"Rows in checkpoint: {reloaded.height}")

print(f"File exists after context: {Path(path).exists()}")

---

## Part 6 — Classes, inheritance & dunders (25 min)

### <mark>Exercise 6a: Build a `DataPipeline` class</mark>

Build a class that wraps a sequence of pipeline steps (functions that take and return a `pl.DataFrame`) and runs them in order.

It should implement:
- `__init__(self, steps)` — store the list of step functions
- `__len__` — return the number of steps
- `__repr__` — return a useful string like `DataPipeline(steps=['remove_nulls', 'add_features'])`
- `run(self, df)` — apply each step in order, log each step name, return the final DataFrame

In [ ]:
class DataPipeline:
    """A simple sequential pipeline of DataFrame transformation steps."""
    
    def __init__(self, steps: list):
        # TODO: store the steps
        pass
    
    def __len__(self) -> int:
        # TODO: return number of steps
        pass
    
    def __repr__(self) -> str:
        # TODO: return something like DataPipeline(steps=['func1', 'func2'])
        # Hint: use func.__name__ to get the name of a function
        pass
    
    def run(self, df: pl.DataFrame) -> pl.DataFrame:
        """Run all steps in order and return the result."""
        # TODO: loop through self.steps, apply each one, log each step name
        pass


# Test it
pipeline = DataPipeline(steps=[
    remove_unknown_outcomes,
    remove_missing_ages,
    add_features,
])

print(pipeline)          # should show step names
print(len(pipeline))     # should be 3

output = pipeline.run(test_df)
output

### <mark>Exercise 6b: A child class — `TimedPipeline`</mark>

Create a child class `TimedPipeline` that inherits from `DataPipeline` and overrides `run()` to also:
- Record how long each step takes
- After all steps complete, print a summary table showing step name and time taken

Use your `timer` context manager from Part 5 if you finished it, otherwise use `time.perf_counter()` directly.

In [ ]:
class TimedPipeline(DataPipeline):
    """A DataPipeline that records and reports the time taken by each step."""
    
    def run(self, df: pl.DataFrame) -> pl.DataFrame:
        """Run all steps and print a timing summary."""
        # TODO:
        # - same as DataPipeline.run, but record elapsed time per step
        # - after all steps, build a small pl.DataFrame with columns ["step", "seconds"]
        #   and print it
        pass


# Test it
timed = TimedPipeline(steps=[
    remove_unknown_outcomes,
    remove_missing_ages,
    add_features,
])

output = timed.run(test_df)

# Expected: same output as before, plus a timing table printed to the console

---

## Part 7 — Putting it all together (20 min)


You now have all the ingredients. This final exercise brings them together into a mini ML-ready pipeline.

### <mark>Exercise 7: Extend `DataPipeline` to support iteration and use-as-context-manager</mark>

Add two more dunder methods to your `DataPipeline`:

**Part A** — Make it iterable with `__iter__` and `__next__` so you can loop over its steps:
```python
for step in pipeline:
    print(step.__name__)
```

**Part B** — Make it a context manager with `__enter__` and `__exit__` so it can be used like:
```python
with DataPipeline(steps=[...]) as pipeline:
    result = pipeline.run(df)
# On exit: log how many steps ran and that the pipeline is done
```

In [ ]:
class DataPipeline:
    """A sequential DataFrame pipeline with logging, iteration, and context manager support."""

    def __init__(self, steps: list):
        self._steps = steps
        self._current = 0  # needed for __next__

    def __len__(self) -> int:
        return len(self._steps)

    def __repr__(self) -> str:
        names = [s.__name__ for s in self._steps]
        return f"DataPipeline(steps={names})"

    def __iter__(self):
        # TODO: reset self._current to 0 and return self
        pass

    def __next__(self):
        # TODO: return the next step, or raise StopIteration if done
        pass

    def __enter__(self):
        # TODO: log that the pipeline is starting, return self
        pass

    def __exit__(self, exc_type, exc_val, exc_tb):
        # TODO: log that the pipeline finished (or failed), return False
        # Returning False means exceptions are NOT suppressed
        pass

    def run(self, df: pl.DataFrame) -> pl.DataFrame:
        """Run all steps in order."""
        for step in self:
            logger.info("Running step: %s", step.__name__)
            df = step(df)
        return df


# Part A — iteration
pipeline = DataPipeline(steps=[remove_unknown_outcomes, remove_missing_ages, add_features])

print("Steps in the pipeline:")
for step in pipeline:
    print(" -", step.__name__)

print()

# Part B — context manager
with DataPipeline(steps=[remove_unknown_outcomes, remove_missing_ages, add_features]) as p:
    result = p.run(test_df)

result

---

## Session complete

Here's what you practised today:

| Pattern | Where you used it |
|---|---|
| Single-responsibility functions | `_check_is_dog`, `_get_sex`, etc. in Part 2 |
| Unit tests + fixtures | Part 3 |
| `logging` + decorators | `log_step` in Part 4 |
| `@contextmanager` with `try/finally` | `timer`, `dataframe_checkpoint` in Part 5 |
| Class with `__len__`, `__repr__` | `DataPipeline` in Part 6 |
| Inheritance | `TimedPipeline(DataPipeline)` in Part 6 |
| `__iter__` / `__next__` | Part 7a |
| `__enter__` / `__exit__` | Part 7b |